# ELT blend: event-loss severity

Weights $w$ for $F_w = \sum_m w_m F_m$ over STORM, CHAZ and IRIS event-loss distributions, fitted to each target's
event sample by minimising the mean score (grid search). Data: `impact/TC_AUS/TC_AUS_impact_<model>.csv`, one row
per event, zero-loss events dropped; every event has equal weight (severity only, no frequency).
**All sections use only losses $X > 1$** (`LOSS_MIN`), in the targets and the components.

| Section | What it runs |
|---|---|
| 1 | every target × standard scores, $X > 1$ |
| 2 | every target × quantile windows, $X > 1$ |
| 3 | 1 and 2 on $1 < X \le u$, with $u$ the largest IBTrACS event loss |
| 4 | standard scores re-fitted on random subsets of events: $X > 1$ and $1 < X \le u$ |
| 5 | sections 1-3 repeated for each 2-model blend (STORM+CHAZ, STORM+IRIS, CHAZ+IRIS) |
| 6 | save all results |

**Rule sets.** `STANDARD`: CRPS, LogScore (discrete, bin width `LOG_BIN_WIDTH`), Quantile (0.05-0.95), a low/high
quantile triple, and QuantileWeightedCRPS. `WINDOWS`: QuantileScore on sliding windows of three quantile levels
(0.05-0.2, 0.1-0.3, ..., 0.8-0.95), showing how the weights move from low to high losses. Two windows coincide with
the STANDARD triples, so `{**STANDARD, **WINDOWS}` holds 13 distinct rules. Lower scores are better.

**Results** columns start `target, scoring_rule, weight_*`. Every result is stored in `results[<name>]`; the last
cell saves them all. Roughly 5-10 s per optimisation at `bc.GRID_STEP = 0.01` (`0.05` for a quick look).

In [9]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from dataclasses import replace
import scoring_rules_core as bc
from scoring_rules_core import *

data = load_impact_dir("impact/TC_AUS")
RUNS_DIR, NOTES = "results/blend_runs_ELT", ""

COMPONENTS = ('STORM', 'CHAZ', 'IRIS')
TARGETS = ['IBTrACS', 'IBTrACS_p', 'WN_MiniTC_repeat', 'WN_MiniTC_perturb', 'WN_MiniTCQM_repeat', 'WN_MiniTCQM_perturb']
LOSS_MIN = 1                     # every blend keeps only losses > LOSS_MIN (targets and components)
MIN_OBS = 5                      # specs with fewer target events are skipped (listed in df.attrs['skipped'])
LOG_BIN_WIDTH = 1e7              # LogScore bin width (loss units)

STANDARD = standard_rules(low_quantiles=(0.7, 0.8, 0.9), log_score_kwargs=dict(bin_width=LOG_BIN_WIDTH))
WINDOWS = quantile_window_rules()
BASE = BlendSpec(TARGETS[0], COMPONENTS, loss_min=LOSS_MIN)
results = {}
summary(data)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
CHAZ                   kept   19352 of  269877 events
IBTrACS                kept     147 of  100188 events
IBTrACS_p              kept    4326 of  103106 events
IRIS                   kept    7218 of  100188 events
STORM                  kept    9873 of  100188 events
WN_MiniTC_perturb      kept    3744 of  100188 events
WN_MiniTC_repeat       kept    3923 of  100188 events
WN_MiniTCQM_perturb    kept    3696 of  100188 events
WN_MiniTCQM_repeat     kept    3875 of  100188 events


,n,mean,median,max,frac_zero,year_min,year_max,n_years
model,,,,,,,,
CHAZ,19352,1.510601e+08,45351.878920,3.256436e+11,0.0,NaN,NaN,0
IBTrACS,147,1.462544e+07,59769.875953,5.972812e+08,0.0,NaN,NaN,0
IBTrACS_p,4326,3.189919e+07,35616.027772,7.472121e+09,0.0,NaN,NaN,0
IRIS,7218,1.409450e+08,507346.336429,5.473890e+10,0.0,NaN,NaN,0
STORM,9873,4.395823e+07,82557.676503,1.427655e+10,0.0,NaN,NaN,0
WN_MiniTCQM_perturb,3696,2.608354e+07,20203.591269,5.004485e+09,0.0,NaN,NaN,0
WN_MiniTCQM_repeat,3875,1.630609e+07,26319.853815,5.004485e+09,0.0,NaN,NaN,0
WN_MiniTC_perturb,3744,2.688342e+07,25293.056087,4.405667e+09,0.0,NaN,NaN,0
WN_MiniTC_repeat,3923,1.718911e+07,31828.318499,4.405667e+09,0.0,NaN,NaN,0


## 1. Standard scores ($X > 1$)

In [ ]:
results['standard'] = sweep(data, expand_specs(BASE, target=TARGETS), STANDARD, min_obs=MIN_OBS)
results['standard']


3/36 (16s)
6/36 (39s)
9/36 (73s)
12/36 (220s)
15/36 (252s)
18/36 (396s)
21/36 (435s)
24/36 (573s)
27/36 (604s)
30/36 (759s)
33/36 (793s)
36/36 (928s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,CRPS,1.00,0.00,0.00,STORM,True,1.407343e+07,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
1,IBTrACS,LogScore,1.00,0.00,0.00,STORM,True,9.940861e-01,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
2,IBTrACS,Quantile,0.88,0.12,0.00,STORM,False,7.192401e+06,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
3,IBTrACS,"Quantile (0.7, 0.8, 0.9)",0.33,0.67,0.00,CHAZ,False,1.115300e+07,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.93,0.07,0.00,STORM,False,1.142786e+07,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
5,IBTrACS,QuantileWeightedCRPS,1.00,0.00,0.00,STORM,True,6.586678e+06,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
6,IBTrACS_p,CRPS,1.00,0.00,0.00,STORM,True,3.160216e+07,STORM+CHAZ+IRIS,All,1 < X,4161,0,9612,18660,7127
7,IBTrACS_p,LogScore,0.64,0.32,0.04,STORM,False,1.997039e+00,STORM+CHAZ+IRIS,All,1 < X,4161,0,9612,18660,7127
8,IBTrACS_p,Quantile,0.74,0.26,0.00,STORM,False,1.616634e+07,STORM+CHAZ+IRIS,All,1 < X,4161,0,9612,18660,7127
9,IBTrACS_p,"Quantile (0.7, 0.8, 0.9)",0.28,0.72,0.00,CHAZ,False,2.575468e+07,STORM+CHAZ+IRIS,All,1 < X,4161,0,9612,18660,7127


## 2. Quantile windows ($X > 1$)

In [ ]:
results['quantile_windows'] = sweep(data, expand_specs(BASE, target=TARGETS), WINDOWS, min_obs=MIN_OBS)
results['quantile_windows']


5/54 (42s)
10/54 (86s)
15/54 (147s)
20/54 (199s)
25/54 (251s)
30/54 (299s)
35/54 (347s)
40/54 (394s)
45/54 (441s)
50/54 (488s)
54/54 (526s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,"Quantile (0.05, 0.1, 0.2)",0.73,0.01,0.26,STORM,False,1.766368e+06,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
1,IBTrACS,"Quantile (0.1, 0.2, 0.3)",0.13,0.47,0.40,CHAZ,False,3.027941e+06,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
2,IBTrACS,"Quantile (0.2, 0.3, 0.4)",0.02,0.73,0.25,CHAZ,False,4.541299e+06,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
3,IBTrACS,"Quantile (0.3, 0.4, 0.5)",0.00,0.85,0.15,CHAZ,False,6.053210e+06,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
4,IBTrACS,"Quantile (0.4, 0.5, 0.6)",0.00,1.00,0.00,CHAZ,True,7.561397e+06,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
5,IBTrACS,"Quantile (0.5, 0.6, 0.7)",0.43,0.57,0.00,CHAZ,False,9.050123e+06,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
6,IBTrACS,"Quantile (0.6, 0.7, 0.8)",0.61,0.37,0.02,STORM,False,1.041230e+07,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
7,IBTrACS,"Quantile (0.7, 0.8, 0.9)",0.33,0.67,0.00,CHAZ,False,1.115300e+07,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
8,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.93,0.07,0.00,STORM,False,1.142786e+07,STORM+CHAZ+IRIS,All,1 < X,142,0,9612,18660,7127
9,IBTrACS_p,"Quantile (0.05, 0.1, 0.2)",0.00,1.00,0.00,CHAZ,True,3.869142e+06,STORM+CHAZ+IRIS,All,1 < X,4161,0,9612,18660,7127


## 3. Conditional on $1 < X < u$, $u$ = first LogScore bin edge above the largest IBTrACS event loss
Target and components are truncated to $1 < X < u$ and each renormalised, so the fit is
$\sum_m w_m F_m(x \mid 1 < X < u)$: the same window for every target. $u$ is the largest IBTrACS event loss rounded
**up to the next bin edge** (a multiple of `LOG_BIN_WIDTH`), so every IBTrACS event is kept and no LogScore bin is cut
part-way: cutting inside a bin removes model events from the bin of the largest observations and distorts its
probability (e.g. STORM's bin probability for the largest IBTrACS event fell 3x when cutting at the event itself).

In [ ]:
U_MAX = data.loc[data['model'] == 'IBTrACS', 'loss'].max()                   # largest IBTrACS event loss
U = float((np.floor(U_MAX / LOG_BIN_WIDTH) + 1) * LOG_BIN_WIDTH)                # next bin edge above it (loss_max is strict)
RULES_3 = {**STANDARD, **WINDOWS}
print(f"largest IBTrACS event = {U_MAX:.4g}, u = {U:.4g}")

results['conditional'] = sweep(data, expand_specs(replace(BASE, loss_max=U), target=TARGETS), RULES_3, min_obs=MIN_OBS)
results['conditional']


u = 5.973e+08
7/78 (46s)
14/78 (107s)
21/78 (331s)
28/78 (398s)
35/78 (590s)
42/78 (664s)
49/78 (864s)
56/78 (934s)
63/78 (1150s)
70/78 (1232s)
77/78 (1447s)
78/78 (1457s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,CRPS,0.74,0.26,0.00,STORM,False,1.375292e+07,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,142,0,9471,18186,6874
1,IBTrACS,LogScore,0.72,0.28,0.00,STORM,False,9.842504e-01,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,142,0,9471,18186,6874
2,IBTrACS,Quantile,0.62,0.38,0.00,STORM,False,7.109402e+06,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,142,0,9471,18186,6874
3,IBTrACS,"Quantile (0.7, 0.8, 0.9)",0.72,0.20,0.08,STORM,False,1.113645e+07,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,142,0,9471,18186,6874
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.57,0.43,0.00,STORM,False,1.089897e+07,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,142,0,9471,18186,6874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,WN_MiniTCQM_perturb,"Quantile (0.2, 0.3, 0.4)",0.00,1.00,0.00,CHAZ,True,4.158585e+06,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,3482,0,9471,18186,6874
74,WN_MiniTCQM_perturb,"Quantile (0.3, 0.4, 0.5)",0.00,1.00,0.00,CHAZ,True,5.544326e+06,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,3482,0,9471,18186,6874
75,WN_MiniTCQM_perturb,"Quantile (0.4, 0.5, 0.6)",0.00,1.00,0.00,CHAZ,True,6.928060e+06,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,3482,0,9471,18186,6874
76,WN_MiniTCQM_perturb,"Quantile (0.5, 0.6, 0.7)",0.00,1.00,0.00,CHAZ,True,8.302861e+06,STORM+CHAZ+IRIS,All,1 < X < 5.97e+08,3482,0,9471,18186,6874


## 4. Subsampling
Re-fits on `N_ITER` random subsets of `FRAC` of each target's events, for $X > 1$ and for $1 < X \le u$ (section 3).
Columns give the mean, sd and 2.5 / 50 / 97.5% of each weight, and the share of subsets with a corner solution.
Events of one storm are not independent, so this understates the true spread.

In [ ]:
FRAC, N_ITER = 0.6, 50
RULES_4 = STANDARD

results['subsample'] = subsample_sweep(data, expand_specs(BASE, target=TARGETS, loss_range=[(LOSS_MIN, None), (LOSS_MIN, U)]),
                                       RULES_4, frac=FRAC, n_iter=N_ITER, by='row', min_obs=MIN_OBS)
results['subsample']


1/72 (480s)
2/72 (494s)
3/72 (860s)
4/72 (1211s)
5/72 (1576s)
6/72 (2028s)
7/72 (2535s)
8/72 (2551s)
9/72 (2905s)
10/72 (3241s)
11/72 (3584s)
12/72 (4019s)
13/72 (4785s)
14/72 (4818s)
15/72 (5398s)
16/72 (5828s)
17/72 (6251s)
18/72 (10682s)
19/72 (11453s)
20/72 (11484s)
21/72 (12115s)
22/72 (12647s)
23/72 (13165s)
24/72 (18578s)
25/72 (19059s)
26/72 (19073s)
27/72 (19400s)
28/72 (19657s)
29/72 (19936s)
30/72 (25280s)
31/72 (26567s)
32/72 (26619s)
33/72 (27187s)
34/72 (27455s)
35/72 (27726s)
36/72 (30721s)
37/72 (31220s)
38/72 (31235s)
39/72 (31569s)
40/72 (31839s)
41/72 (32111s)
42/72 (35017s)
43/72 (35503s)
44/72 (35518s)
45/72 (35840s)
46/72 (36104s)
47/72 (36367s)
48/72 (39286s)
49/72 (39787s)
50/72 (39802s)
51/72 (40141s)
52/72 (40413s)
53/72 (40684s)
54/72 (43695s)
55/72 (44187s)
56/72 (44203s)
57/72 (44526s)
58/72 (44787s)
59/72 (45055s)
60/72 (48113s)
61/72 (48611s)
62/72 (48625s)
63/72 (48957s)
64/72 (49226s)
65/72 (49507s)
66/72 (52355s)
67/72 (52835s)
68/72 (52849s)
69/72 (53

,target,scoring_rule,weight_STORM_mean,weight_STORM_sd,weight_STORM_q025,weight_STORM_median,weight_STORM_q975,weight_CHAZ_mean,weight_CHAZ_sd,weight_CHAZ_q025,...,weight_IRIS_sd,weight_IRIS_q025,weight_IRIS_median,weight_IRIS_q975,frac_at_corner,components,loss_range,by,frac,n_iter
0,IBTrACS,CRPS,1.0000,0.000000,1.00000,1.00,1.00000,0.0000,0.000000,0.00000,...,0.000000,0.0,0.00,0.00000,1.00,STORM+CHAZ+IRIS,1 < X,row,0.6,50
1,IBTrACS,LogScore,0.9386,0.159143,0.44175,1.00,1.00000,0.0614,0.159143,0.00000,...,0.000000,0.0,0.00,0.00000,0.80,STORM+CHAZ+IRIS,1 < X,row,0.6,50
2,IBTrACS,Quantile,0.8244,0.212787,0.33000,0.88,1.00000,0.1756,0.212787,0.00000,...,0.000000,0.0,0.00,0.00000,0.14,STORM+CHAZ+IRIS,1 < X,row,0.6,50
3,IBTrACS,"Quantile (0.7, 0.8, 0.9)",0.3658,0.389296,0.00000,0.23,0.98000,0.6242,0.387488,0.02000,...,0.028997,0.0,0.00,0.09775,0.38,STORM+CHAZ+IRIS,1 < X,row,0.6,50
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.9050,0.124035,0.46000,0.93,1.00000,0.0950,0.124035,0.00000,...,0.000000,0.0,0.00,0.00000,0.06,STORM+CHAZ+IRIS,1 < X,row,0.6,50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,WN_MiniTCQM_perturb,LogScore,0.4336,0.064831,0.30675,0.43,0.54000,0.5632,0.061923,0.46000,...,0.009355,0.0,0.00,0.03775,0.00,STORM+CHAZ+IRIS,1 < X < 5.97e+08,row,0.6,50
68,WN_MiniTCQM_perturb,Quantile,0.2078,0.107310,0.01000,0.22,0.40325,0.7922,0.107310,0.59675,...,0.000000,0.0,0.00,0.00000,0.02,STORM+CHAZ+IRIS,1 < X < 5.97e+08,row,0.6,50
69,WN_MiniTCQM_perturb,"Quantile (0.7, 0.8, 0.9)",0.0882,0.106956,0.00000,0.03,0.34550,0.8240,0.075485,0.63900,...,0.056326,0.0,0.09,0.18000,0.00,STORM+CHAZ+IRIS,1 < X < 5.97e+08,row,0.6,50
70,WN_MiniTCQM_perturb,"Quantile (0.8, 0.9, 0.95)",0.2066,0.118797,0.00000,0.23,0.41100,0.7934,0.118797,0.58900,...,0.000000,0.0,0.00,0.00000,0.10,STORM+CHAZ+IRIS,1 < X < 5.97e+08,row,0.6,50


## 5. Model pairs
Every 2-model blend (STORM+CHAZ, STORM+IRIS, CHAZ+IRIS) refitted for every target: standard scores, quantile
windows, and the section 3 conditional ($1 < X \le u$). Results go to `results['pairs_standard']`,
`results['pairs_quantile_windows']` and `results['pairs_conditional']` and are saved with everything else.
`min_mean_score` is directly comparable with the 3-model run for the same target, rule and condition, so the
difference is what dropping a model costs (the viewer tabulates this).

In [ ]:
PAIRS = model_subsets(COMPONENTS, sizes=(2,))      # [('STORM', 'CHAZ'), ('STORM', 'IRIS'), ('CHAZ', 'IRIS')]
print(PAIRS)

results['pairs_standard'] = sweep(data, expand_specs(BASE, target=TARGETS, components=PAIRS), STANDARD, min_obs=MIN_OBS)
results['pairs_standard']


[('STORM', 'CHAZ'), ('STORM', 'IRIS'), ('CHAZ', 'IRIS')]
10/108 (2s)
20/108 (4s)
30/108 (12s)
40/108 (18s)
50/108 (25s)
60/108 (33s)
70/108 (39s)
80/108 (47s)
90/108 (55s)
100/108 (61s)
108/108 (68s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,CRPS,1.00,0.00,NaN,STORM,True,1.407343e+07,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
1,IBTrACS,LogScore,1.00,0.00,NaN,STORM,True,9.940861e-01,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
2,IBTrACS,Quantile,0.88,0.12,NaN,STORM,False,7.192401e+06,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
3,IBTrACS,"Quantile (0.7, 0.8, 0.9)",0.33,0.67,NaN,CHAZ,False,1.115300e+07,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.93,0.07,NaN,STORM,False,1.142786e+07,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,WN_MiniTCQM_perturb,LogScore,NaN,0.79,0.21,CHAZ,False,2.110481e+00,CHAZ+IRIS,All,1 < X,3514,0,NaN,18660.0,7127.0
104,WN_MiniTCQM_perturb,Quantile,NaN,1.00,0.00,CHAZ,True,1.343420e+07,CHAZ+IRIS,All,1 < X,3514,0,NaN,18660.0,7127.0
105,WN_MiniTCQM_perturb,"Quantile (0.7, 0.8, 0.9)",NaN,1.00,0.00,CHAZ,True,2.137073e+07,CHAZ+IRIS,All,1 < X,3514,0,NaN,18660.0,7127.0
106,WN_MiniTCQM_perturb,"Quantile (0.8, 0.9, 0.95)",NaN,1.00,0.00,CHAZ,True,2.266681e+07,CHAZ+IRIS,All,1 < X,3514,0,NaN,18660.0,7127.0


In [ ]:
results['pairs_quantile_windows'] = sweep(data, expand_specs(BASE, target=TARGETS, components=PAIRS), WINDOWS,
                                          min_obs=MIN_OBS)
results['pairs_quantile_windows']


16/162 (4s)
32/162 (8s)
48/162 (13s)
64/162 (18s)
80/162 (23s)
96/162 (28s)
112/162 (33s)
128/162 (37s)
144/162 (42s)
160/162 (46s)
162/162 (46s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,"Quantile (0.05, 0.1, 0.2)",1.00,0.00,NaN,STORM,True,1.766368e+06,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
1,IBTrACS,"Quantile (0.1, 0.2, 0.3)",0.00,1.00,NaN,CHAZ,True,3.027969e+06,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
2,IBTrACS,"Quantile (0.2, 0.3, 0.4)",1.00,0.00,NaN,STORM,True,4.541353e+06,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
3,IBTrACS,"Quantile (0.3, 0.4, 0.5)",0.59,0.41,NaN,STORM,False,6.053257e+06,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
4,IBTrACS,"Quantile (0.4, 0.5, 0.6)",0.00,1.00,NaN,CHAZ,True,7.561397e+06,STORM+CHAZ,All,1 < X,142,0,9612.0,18660.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,WN_MiniTCQM_perturb,"Quantile (0.4, 0.5, 0.6)",NaN,1.00,0.00,CHAZ,True,1.371441e+07,CHAZ+IRIS,All,1 < X,3514,0,NaN,18660.0,7127.0
158,WN_MiniTCQM_perturb,"Quantile (0.5, 0.6, 0.7)",NaN,1.00,0.00,CHAZ,True,1.644595e+07,CHAZ+IRIS,All,1 < X,3514,0,NaN,18660.0,7127.0
159,WN_MiniTCQM_perturb,"Quantile (0.6, 0.7, 0.8)",NaN,0.99,0.01,CHAZ,False,1.911931e+07,CHAZ+IRIS,All,1 < X,3514,0,NaN,18660.0,7127.0
160,WN_MiniTCQM_perturb,"Quantile (0.7, 0.8, 0.9)",NaN,1.00,0.00,CHAZ,True,2.137073e+07,CHAZ+IRIS,All,1 < X,3514,0,NaN,18660.0,7127.0


In [ ]:
results['pairs_conditional'] = sweep(data, expand_specs(replace(BASE, loss_max=U), target=TARGETS, components=PAIRS),
                                     RULES_3, min_obs=MIN_OBS)
results['pairs_conditional']


23/234 (3s)
46/234 (9s)
69/234 (16s)
92/234 (24s)
115/234 (31s)
138/234 (39s)
161/234 (45s)
184/234 (52s)
207/234 (60s)
230/234 (67s)
234/234 (68s)


,target,scoring_rule,weight_STORM,weight_CHAZ,weight_IRIS,dominant_model,at_corner,min_mean_score,components,enso_phase,loss_range,n_obs,n_above_dropped,n_events_STORM,n_events_CHAZ,n_events_IRIS
0,IBTrACS,CRPS,0.74,0.26,NaN,STORM,False,1.375292e+07,STORM+CHAZ,All,1 < X < 5.97e+08,142,0,9471.0,18186.0,NaN
1,IBTrACS,LogScore,0.72,0.28,NaN,STORM,False,9.842504e-01,STORM+CHAZ,All,1 < X < 5.97e+08,142,0,9471.0,18186.0,NaN
2,IBTrACS,Quantile,0.62,0.38,NaN,STORM,False,7.109402e+06,STORM+CHAZ,All,1 < X < 5.97e+08,142,0,9471.0,18186.0,NaN
3,IBTrACS,"Quantile (0.7, 0.8, 0.9)",0.95,0.05,NaN,STORM,False,1.113720e+07,STORM+CHAZ,All,1 < X < 5.97e+08,142,0,9471.0,18186.0,NaN
4,IBTrACS,"Quantile (0.8, 0.9, 0.95)",0.57,0.43,NaN,STORM,False,1.089897e+07,STORM+CHAZ,All,1 < X < 5.97e+08,142,0,9471.0,18186.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229,WN_MiniTCQM_perturb,"Quantile (0.2, 0.3, 0.4)",NaN,1.00,0.00,CHAZ,True,4.158585e+06,CHAZ+IRIS,All,1 < X < 5.97e+08,3482,0,NaN,18186.0,6874.0
230,WN_MiniTCQM_perturb,"Quantile (0.3, 0.4, 0.5)",NaN,1.00,0.00,CHAZ,True,5.544326e+06,CHAZ+IRIS,All,1 < X < 5.97e+08,3482,0,NaN,18186.0,6874.0
231,WN_MiniTCQM_perturb,"Quantile (0.4, 0.5, 0.6)",NaN,1.00,0.00,CHAZ,True,6.928060e+06,CHAZ+IRIS,All,1 < X < 5.97e+08,3482,0,NaN,18186.0,6874.0
232,WN_MiniTCQM_perturb,"Quantile (0.5, 0.6, 0.7)",NaN,1.00,0.00,CHAZ,True,8.302861e+06,CHAZ+IRIS,All,1 < X < 5.97e+08,3482,0,NaN,18186.0,6874.0


In [ ]:
# Optional: subsampling for the pairs (same settings as section 4); uncomment to run.
# results['pairs_subsample'] = subsample_sweep(data, expand_specs(BASE, target=TARGETS, components=PAIRS), RULES_4,
#                                              frac=FRAC, n_iter=N_ITER, by='row', min_obs=MIN_OBS)


## 6. Save all results

In [14]:
for tag, df in results.items():
    save_run(df, tag, out_dir=RUNS_DIR, notes=NOTES)
pd.read_csv(f'{RUNS_DIR}/manifest.csv').tail(len(results))


saved 108 rows -> results\blend_runs_ELT\20261030_014841__pairs_standard
saved 162 rows -> results\blend_runs_ELT\20261030_014842__pairs_quantile_windows
saved 234 rows -> results\blend_runs_ELT\20261030_014842__pairs_conditional


,timestamp,tag,n_rows,folder,notes
9,20261030_014841,pairs_standard,108,results\blend_runs_ELT\20261030_014841__pairs_...,NaN
10,20261030_014842,pairs_quantile_windows,162,results\blend_runs_ELT\20261030_014842__pairs_...,NaN
11,20261030_014842,pairs_conditional,234,results\blend_runs_ELT\20261030_014842__pairs_...,NaN
